# Fetch Report From an Already-Completed Workflow Run

**Scope, deliberately minimal:**
- Workflow is triggered **manually** (by you, in GitHub) — this notebook never triggers anything
- This notebook **only fetches** the report from a run that already finished
- **No Artifactory upload.** The report lives only as a GitHub Actions build artifact — nothing
  here pushes it anywhere else
- **No webhook, no server.** Just run this notebook after the workflow finishes and it pulls
  the report directly via the GitHub API

Two ways to point it at a run:
- Leave `RUN_ID = None` → it grabs the **most recent completed & successful** run of the workflow
- Set `RUN_ID` to a specific number → it fetches that exact run (useful if several ran recently)


---
## Phase 1 — Configuration


In [ ]:
import os, io, json, zipfile, logging, dataclasses
from dataclasses import dataclass
from typing import Optional
import requests

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger("fetch-report")

GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN", "")                                # already provided
GITHUB_REPO = os.environ.get("GITHUB_REPO", "org/repo")                          # TODO: your real org/repo
GITHUB_WORKFLOW_FILE = os.environ.get("GITHUB_WORKFLOW_FILE", "prisma-scan.yml") # TODO: your real workflow filename
ARTIFACT_NAME_HINT = os.environ.get("PRISMA_ARTIFACT_NAME", "prisma-scan-report") # TODO: confirm exact artifact name

RUN_ID: Optional[int] = None   # set to a specific run number to target it, or leave None for "latest"

GITHUB_API = "https://api.github.com"

if not GITHUB_TOKEN:
    log.warning("GITHUB_TOKEN not set -- running in DRY-RUN mode with bundled demo data "
                "so the notebook still runs top to bottom.")

def _gh_headers() -> dict:
    return {
        "Authorization": f"Bearer {GITHUB_TOKEN}",
        "Accept": "application/vnd.github+json",
        "X-GitHub-Api-Version": "2022-11-28",
    }


---
## Phase 2 — Find the Run

If `RUN_ID` is set, fetches that run directly and confirms it's actually completed and
successful. Otherwise, lists recent runs of the workflow and picks the most recent one that
completed successfully — it deliberately does **not** just grab the most recent run regardless
of outcome, since a failed run has no usable report.


In [ ]:
class RunNotFoundError(RuntimeError):
    pass

class RunNotSuccessfulError(RuntimeError):
    pass


def find_run(run_id: Optional[int] = None) -> dict:
    if not GITHUB_TOKEN:
        log.info("[DRY-RUN] Would look up %s", f"run_id={run_id}" if run_id else "latest completed run")
        return {"id": run_id or 0, "status": "completed", "conclusion": "success", "dry_run": True}

    if run_id is not None:
        url = f"{GITHUB_API}/repos/{GITHUB_REPO}/actions/runs/{run_id}"
        resp = requests.get(url, headers=_gh_headers(), timeout=30)
        resp.raise_for_status()
        run = resp.json()
    else:
        url = f"{GITHUB_API}/repos/{GITHUB_REPO}/actions/workflows/{GITHUB_WORKFLOW_FILE}/runs"
        resp = requests.get(url, headers=_gh_headers(),
                             params={"status": "completed", "per_page": 5}, timeout=30)
        resp.raise_for_status()
        runs = resp.json().get("workflow_runs", [])
        if not runs:
            raise RunNotFoundError(f"No completed runs found for {GITHUB_WORKFLOW_FILE}. "
                                    f"Has it been triggered manually yet?")
        successful = [r for r in runs if r["conclusion"] == "success"]
        if not successful:
            most_recent = runs[0]
            raise RunNotSuccessfulError(
                f"Most recent completed run (id={most_recent['id']}) concluded as "
                f"'{most_recent['conclusion']}', not 'success' -- no usable report to fetch. "
                f"Trigger the workflow again, or pass a specific successful RUN_ID."
            )
        run = successful[0]

    log.info("Using run id=%s status=%s conclusion=%s", run["id"], run["status"], run["conclusion"])
    return run


run = find_run(RUN_ID)
run


---
## Phase 3 — Download the Artifact


In [ ]:
def list_artifacts(run_id: int) -> list[dict]:
    if not GITHUB_TOKEN:
        log.info("[DRY-RUN] Would list artifacts for run_id=%s", run_id)
        return [{"id": 0, "name": ARTIFACT_NAME_HINT}]
    url = f"{GITHUB_API}/repos/{GITHUB_REPO}/actions/runs/{run_id}/artifacts"
    resp = requests.get(url, headers=_gh_headers(), timeout=30)
    resp.raise_for_status()
    return resp.json().get("artifacts", [])


def download_artifact_zip(artifact_id: int) -> Optional[bytes]:
    if not GITHUB_TOKEN:
        log.info("[DRY-RUN] Would download artifact_id=%s -- using bundled demo report instead", artifact_id)
        return None
    url = f"{GITHUB_API}/repos/{GITHUB_REPO}/actions/artifacts/{artifact_id}/zip"
    resp = requests.get(url, headers=_gh_headers(), timeout=60)
    resp.raise_for_status()
    return resp.content


def extract_json_report(zip_bytes: bytes) -> dict:
    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zf:
        json_names = [n for n in zf.namelist() if n.endswith(".json")]
        if not json_names:
            raise FileNotFoundError(f"No .json file in artifact. Contents: {zf.namelist()}")
        with zf.open(json_names[0]) as f:
            return json.load(f)


artifacts = list_artifacts(run["id"])
matching = [a for a in artifacts if ARTIFACT_NAME_HINT in a["name"]] or artifacts
if not matching:
    raise RuntimeError(f"No artifacts found on run {run['id']}")
target_artifact = matching[0]
log.info("Using artifact: %s", target_artifact)

zip_bytes = download_artifact_zip(target_artifact["id"])


---
## Phase 4 — Parse the Report

Bundled demo report used automatically in dry-run mode, so this cell always produces real,
inspectable output.


In [ ]:
SEVERITY_ORDER = {"critical": 4, "high": 3, "medium": 2, "low": 1, "unimportant": 0}

@dataclass
class Vulnerability:
    cve_id: str
    severity: str
    package_name: str
    package_version: str
    fix_version: Optional[str]
    description: str

    @property
    def rank(self) -> int:
        return SEVERITY_ORDER.get(self.severity.lower(), 0)


def parse_prisma_report(report: dict) -> list[Vulnerability]:
    vulns = []
    for result in report.get("results", []):
        for v in result.get("vulnerabilities", []) or []:
            vulns.append(Vulnerability(
                cve_id=v.get("id", "UNKNOWN"), severity=v.get("severity", "unimportant"),
                package_name=v.get("packageName", "unknown"), package_version=v.get("packageVersion", "unknown"),
                fix_version=v.get("fixDate") or v.get("status"), description=v.get("description", ""),
            ))
    vulns.sort(key=lambda v: v.rank, reverse=True)
    return vulns


DEMO_PRISMA_REPORT = {
    "results": [{
        "name": "payments-api:1.4.2",
        "vulnerabilities": [
            {"id": "CVE-2024-6119", "severity": "critical", "packageName": "openssl",
             "packageVersion": "3.0.2", "fixDate": "3.0.13",
             "description": "OpenSSL denial of service via crafted X.509 certificate."},
            {"id": "CVE-2023-44487", "severity": "high", "packageName": "nghttp2",
             "packageVersion": "1.43.0", "fixDate": "1.57.0",
             "description": "HTTP/2 Rapid Reset DoS."},
            {"id": "CVE-2022-37434", "severity": "medium", "packageName": "zlib",
             "packageVersion": "1.2.11", "fixDate": "1.2.12",
             "description": "Heap buffer over-read in inflate()."},
        ],
    }],
}

report = extract_json_report(zip_bytes) if zip_bytes is not None else DEMO_PRISMA_REPORT
vulns = parse_prisma_report(report)

for v in vulns:
    print(f"{v.severity:8s} {v.cve_id:16s} {v.package_name} {v.package_version} -> fix {v.fix_version}")


---
## Phase 5 — Output as JSON

This is the deliverable: the report, as JSON, in a Python variable and saved to disk.


In [ ]:
report_json = json.dumps([dataclasses.asdict(v) for v in vulns], indent=2)

with open("vulnerability_report.json", "w") as f:
    f.write(report_json)

log.info("Saved %d vulnerabilities to vulnerability_report.json", len(vulns))
print(report_json)


---
## To run this for real

1. Trigger the workflow manually in GitHub (Actions tab → your workflow → Run workflow)
2. Wait for it to finish
3. Fill in `GITHUB_REPO`, `GITHUB_WORKFLOW_FILE`, and `ARTIFACT_NAME_HINT` in Phase 1
4. Make sure `GITHUB_TOKEN` is set in the environment
5. Run all cells — `report_json` at the end is your answer

Confirm `ARTIFACT_NAME_HINT` matches your workflow's `actions/upload-artifact` step name before
running live — that's the one value most likely to need adjusting.
